<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: From Single Agent to Multi-Agent Workflow </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Demonstrate the transition from a **single agent** to a **multi-agent workflow**.  
This notebook shows why single agents have limitations and how splitting responsibilities into specialized agents with basic coordination improves performance and clarity.

**Key Topics Covered Today:**
- Limitations of single-agent systems
- Benefits of role specialization
- Basic task delegation pattern
- Simple inter-agent communication
- From monolithic agent → Multi-agent team

---

## 1. Setup

In [1]:
# Run this cell first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" duckduckgo-search --force-reinstall
!pip install -U ddgs

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2

## 2. Single Agent Workflow (Review from Week 3)

Let's start with a single agent that handles the entire travel budget planning task.

In [3]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langsmith import Client

client = Client()

search_tool = DuckDuckGoSearchRun(name="Web_Search")

@tool
def calculate_math(expression: str) -> str:
    """Perform math calculations (addition, multiplication, percentages, etc.)"""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: {str(e)}"

tools = [search_tool, calculate_math]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

single_agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

single_executor = AgentExecutor(
    agent=single_agent,
    tools=tools,
    verbose=True,
    max_iterations=10
)

print("Single Agent Ready")

Single Agent Ready


## 3. Demo: Single Agent (Monolithic Approach)

In [4]:
single_query = """Plan a budget trip from Riyadh to Istanbul.
Find approximate cheapest round-trip flight price in USD,
average hotel price per night in USD, current USD to SAR exchange rate,
then calculate total budget in SAR for flights + 7 nights hotel,
and finally calculate what percentage the flight cost is of the total budget."""

response_single = single_executor.invoke({"input": single_query})

print("\n" + "="*70)
print("SINGLE AGENT FINAL ANSWER:")
print(response_single["output"])



> Entering new AgentExecutor chain...
I need to gather information on the following: the approximate cheapest round-trip flight price from Riyadh to Istanbul in USD, the average hotel price per night in Istanbul in USD, and the current USD to SAR exchange rate. After that, I will calculate the total budget in SAR for flights and 7 nights of hotel accommodation, and finally determine what percentage the flight cost is of the total budget.

Action: Web_Search  
Action Input: "cheapest round-trip flight price from Riyadh to Istanbul"  June 16, 2025 - The cheapest flights from Riyadh to Istanbul with Aviasales.ge starting at 200 GEL. Compare airfares for the route Riyadh (RUH) – Istanbul (IST), sales, and discounts. April 2, 2026 - Cheap flights from Riyadh to Islamabad ✈︎ Compare over 1000+ booking sites ✓ Find the lowest price ✓ Fast & easy booking ✓ Visit www.wego.com now. March 30, 2026 - Flights between Istanbul and Riyadh starting at 72 €. Choose between Turkish Airlines, Flyadeal,

## 4. Why Move to Multi-Agent Systems?

**Limitations of Single Agent:**
- Does everything → becomes slow and unfocused
- Hard to debug when something goes wrong
- One agent must be expert in many domains
- Poor separation of concerns
- Harder to scale or add new capabilities

**Benefits of Multi-Agent:**
- Role specialization (Researcher, Analyst, Calculator, Supervisor)
- Better reasoning per role
- Easier to trace and debug
- Natural task delegation
- More modular and maintainable

---

## 5. Building a Simple Multi-Agent Workflow

### Creating Specialized Agents

We will create two specialized agents:
1. **Research Agent** — Good at searching and gathering information
2. **Math Agent** — Good at performing calculations and percentages

In [6]:
# Research Agent (focuses only on searching)
research_prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)
research_agent = create_react_agent(llm=llm, tools=[search_tool], prompt=research_prompt)

research_executor = AgentExecutor(
    agent=research_agent,
    tools=[search_tool],
    verbose=False,   # Set to True if you want to see details
    max_iterations=6
)

# Math Agent (focuses only on calculations)
math_prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)
math_agent = create_react_agent(llm=llm, tools=[calculate_math], prompt=math_prompt)

math_executor = AgentExecutor(
    agent=math_agent,
    tools=[calculate_math],
    verbose=False,
    max_iterations=5
)

print("Research Agent and Math Agent created")

Research Agent and Math Agent created


## 6. Basic Task Delegation Pattern

A simple **Supervisor → Worker** pattern:
- Supervisor decides what to do
- Delegates research to Research Agent
- Delegates calculations to Math Agent
- Combines results

In [7]:
def multi_agent_travel_planner(query: str):
    print("Supervisor: Breaking down the task...\n")

    # Step 1: Research Agent gathers data
    print("Delegating to Research Agent...")
    research_result = research_executor.invoke({
        "input": "Find current cheapest round-trip flight price Riyadh to Istanbul in USD, "
                 "average hotel price per night in Istanbul in USD, "
                 "and current USD to SAR exchange rate."
    })

    research_data = research_result["output"]
    print("Research complete.\n")

    # Step 2: Math Agent performs calculations
    print("Delegating to Math Agent...")
    math_input = f"Using this data: {research_data}\n\nCalculate: total budget in SAR for round-trip flight + 7 nights hotel, " \
                 "and what percentage the flight cost is of the total budget."

    math_result = math_executor.invoke({"input": math_input})

    print("Calculations complete.\n")

    # Final Answer
    final_answer = f"""MULTI-AGENT RESULT

Research Findings:
{research_data}

Calculation Results:
{math_result["output"]}

 Task completed through coordinated agents."""

    return final_answer

# Run the multi-agent workflow
multi_result = multi_agent_travel_planner("Plan budget trip from Riyadh to Istanbul")

print("="*70)
print("MULTI-AGENT FINAL OUTPUT:")
print(multi_result)

Supervisor: Breaking down the task...

Delegating to Research Agent...
Research complete.

Delegating to Math Agent...
Calculations complete.

MULTI-AGENT FINAL OUTPUT:
MULTI-AGENT RESULT

Research Findings:
Agent stopped due to iteration limit or time limit.

Calculation Results:
The total budget is 6500 SAR, and the flight cost is approximately 46.15% of the total budget.

 Task completed through coordinated agents.


## 7. Reflection Questions

1. What were the main limitations you observed in the single agent?
2. How did role specialization help in the multi-agent version?
3. What kind of communication happened between agents?
4. In what real-world scenarios would multi-agent systems be more beneficial?
5. What challenges might arise when scaling to 4–5 specialized agents?
